# Simulated Annealing for Supply Chain Logistics Optimization

This notebook implements a Simulated Annealing metaheuristic for the supply chain logistics optimization problem.

The objective is to minimize total logistics cost while satisfying the relevant supply chain constraints, including product–plant compatibility, plant–port connectivity, plant capacity, and transportation feasibility.

The same optimization problem and processed data will be used for comparison with the MILP approach.

In [1]:
import pandas as pd
import numpy as np
import random
import math
import time

import matplotlib.pyplot as plt

In [3]:
data = pd.read_excel(
    "data/Supply chain logistics problem.xlsx",
    sheet_name=None
)

print("Data loaded successfully.")
print("Sheets:", list(data.keys()))

Data loaded successfully.
Sheets: ['OrderList', 'FreightRates', 'WhCosts', 'WhCapacities', 'ProductsPerPlant', 'VmiCustomers', 'PlantPorts']


In [4]:
orders = data["OrderList"]
freight_rates = data["FreightRates"]
wh_costs = data["WhCosts"]
wh_capacities = data["WhCapacities"]
products_plant = data["ProductsPerPlant"]
vmi_customers = data["VmiCustomers"]
plant_ports = data["PlantPorts"]

print("DataFrames created successfully.")

DataFrames created successfully.


In [5]:
print(orders[
    [
        "Order ID",
        "Origin Port",
        "Carrier",
        "Service Level",
        "Customer",
        "Product ID",
        "Plant Code",
        "Destination Port",
        "Unit quantity",
        "Weight"
    ]
].head())

       Order ID Origin Port Carrier Service Level   Customer  Product ID  \
0  1.447296e+09      PORT09   V44_3           CRF  V55555_53     1700106   
1  1.447158e+09      PORT09   V44_3           CRF  V55555_53     1700106   
2  1.447139e+09      PORT09   V44_3           CRF  V55555_53     1700106   
3  1.447364e+09      PORT09   V44_3           CRF  V55555_53     1700106   
4  1.447364e+09      PORT09   V44_3           CRF  V55555_53     1700106   

  Plant Code Destination Port  Unit quantity  Weight  
0    PLANT16           PORT09            808   14.30  
1    PLANT16           PORT09           3188   87.94  
2    PLANT16           PORT09           2331   61.20  
3    PLANT16           PORT09            847   16.16  
4    PLANT16           PORT09           2163   52.34  


In [6]:
product_to_plants = (
    products_plant
    .groupby("Product ID")["Plant Code"]
    .apply(list)
    .to_dict()
)

print("Number of products:", len(product_to_plants))

# Show a few examples
for product, plants in list(product_to_plants.items())[:10]:
    print(product, "→", plants)

Number of products: 1540
1613321 → ['PLANT03']
1615677 → ['PLANT03']
1617714 → ['PLANT12']
1617826 → ['PLANT12']
1620510 → ['PLANT03']
1620552 → ['PLANT03']
1621717 → ['PLANT12']
1621719 → ['PLANT12', 'PLANT13']
1622685 → ['PLANT03']
1623991 → ['PLANT03']


In [7]:
ordered_products = orders["Product ID"].unique()

ordered_product_to_plants = {
    product: product_to_plants.get(product, [])
    for product in ordered_products
}

print("Products appearing in orders:", len(ordered_products))

# Check whether every ordered product has at least one feasible plant
products_without_plants = [
    product
    for product, plants in ordered_product_to_plants.items()
    if len(plants) == 0
]

print("Products without feasible plants:", len(products_without_plants))

Products appearing in orders: 772
Products without feasible plants: 0


In [10]:
#Plant → Feasible Ports

plant_to_ports = (
    plant_ports
    .groupby("Plant Code")["Port"]
    .apply(list)
    .to_dict()
)

print("Number of plants:", len(plant_to_ports))

for plant, ports in plant_to_ports.items():
    print(plant, "→", ports)

Number of plants: 19
PLANT01 → ['PORT01', 'PORT02']
PLANT02 → ['PORT03']
PLANT03 → ['PORT04']
PLANT04 → ['PORT05']
PLANT05 → ['PORT06']
PLANT06 → ['PORT06']
PLANT07 → ['PORT01', 'PORT02']
PLANT08 → ['PORT04']
PLANT09 → ['PORT04']
PLANT10 → ['PORT01', 'PORT02']
PLANT11 → ['PORT04']
PLANT12 → ['PORT04']
PLANT13 → ['PORT04']
PLANT14 → ['PORT07']
PLANT15 → ['PORT08']
PLANT16 → ['PORT09']
PLANT17 → ['PORT10']
PLANT18 → ['PORT11']
PLANT19 → ['PORT04']
